# 04 - Feature Relationships

**Milestone 1 — Part 4.A**: Investigate correlations, F-statistics, and feature selection.

## Objectives
- Compute pairwise correlation matrix and heatmap
- Compute F-statistic (f_regression) for all features vs target
- Run forward and backward sequential feature selection
- Identify strongly correlated feature pairs

## Expected Outcomes
| Deliverable | Description |
|---|---|
| Correlation heatmap | Visual of pairwise correlations |
| F-statistic bar chart | Feature importance ranking via f_regression |
| Forward selection result | Top-k features chosen by forward SFS |
| Backward selection result | Top-k features chosen by backward SFS |
| Comparison notes | Agreement/disagreement across methods |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_regression, SelectKBest, SequentialFeatureSelector
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TARGET = "taxvaluedollarcnt"

df = pd.read_csv("zillow_cleaned.csv")
print(f"Shape: {df.shape}")

## 1. Correlation Matrix

In [ ]:
corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(20, 16))
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm", center=0, linewidths=0.3)
plt.title("Pairwise Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with the target, sorted
target_corr = corr_matrix[TARGET].drop(TARGET).sort_values(ascending=False)
print("Correlation with target:")
print(target_corr)

## 2. F-Statistic (f_regression)

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

f_scores, p_values = f_regression(X, y)

f_df = pd.DataFrame({"feature": X.columns, "f_score": f_scores, "p_value": p_values})
f_df = f_df.sort_values("f_score", ascending=False).reset_index(drop=True)

plt.figure(figsize=(12, 8))
plt.barh(f_df["feature"], f_df["f_score"], color="steelblue")
plt.xlabel("F-Score")
plt.title("F-Statistic for All Features vs Target")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

f_df.head(15)

## 3. Forward and Backward Feature Selection

Using `SequentialFeatureSelector` with a simple `LinearRegression` estimator.
Adjust `n_features_to_select` as appropriate.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

estimator = LinearRegression()
n_select = 10  # TODO: adjust

# Forward selection
sfs_forward = SequentialFeatureSelector(
    estimator, n_features_to_select=n_select, direction="forward", cv=3, scoring="r2"
)
sfs_forward.fit(X_scaled, y)
forward_features = X.columns[sfs_forward.get_support()].tolist()
print(f"Forward selection ({n_select}): {forward_features}")

In [ ]:
# Backward selection
sfs_backward = SequentialFeatureSelector(
    estimator, n_features_to_select=n_select, direction="backward", cv=3, scoring="r2"
)
sfs_backward.fit(X_scaled, y)
backward_features = X.columns[sfs_backward.get_support()].tolist()
print(f"Backward selection ({n_select}): {backward_features}")

In [ ]:
# Compare the two methods
common = set(forward_features) & set(backward_features)
only_forward = set(forward_features) - set(backward_features)
only_backward = set(backward_features) - set(forward_features)

print(f"Common to both:  {common}")
print(f"Only forward:    {only_forward}")
print(f"Only backward:   {only_backward}")

---
## Discussion 4.A

Describe what you see in the feature relationships and correlations:
- Strongly correlated feature pairs?
- Do correlation, F-statistic, and SFS methods agree or disagree?
- Any surprises?

*YOUR ANSWER HERE*

---
### Next Notebook → `05_bivariate_analysis.ipynb`